What is LangSmith?
LangSmith is a platform developed by the LangChain team for:

Debugging,Testing,Evaluating,Monitoring

LangChain / LangGraph = Development framework
LangSmith = Observability + Evaluation platform

Key Value of LangSmith

Visibility → See every step of execution
Debuggability → Find exact failure points
Evaluability → Measure quality systematically
Observability → Monitor production systems
Iteration Speed → Improve prompts/chains faster

LangSmith Account Setup & API Keys
1. Create LangSmith Account

Go to: https://smith.langchain.com
Sign up using:
Email
Google
GitHub

After login, you will land on the LangSmith dashboard.


2. Create an API Key

Click on your profile (bottom left or top right),
Go to Settings,
Open API Keys section,
Click Create API Key,
Give it a name (example: local-dev),
Copy the key immediately (you won’t see it again)

Your LangChain / LangGraph code runs on your computer or server.
For your code to send data to LangSmith, it needs permission and configuration.
That is why environment variables setup are required.

3. Set Environment Variables

LangSmith needs these environment variables:

$env:LANGCHAIN_TRACING_V2="true"  //Enables tracing

$env:LANGCHAIN_API_KEY="your_langsmith_api_key"  //Authenticates with LangSmith

$env:LANGCHAIN_PROJECT="my-first-project"  //Groups traces under a project name

Then load it in Python:

from dotenv import load_dotenv
load_dotenv()

What is a Run?

A Run is the smallest unit of work recorded in LangSmith.
Every individual action your application performs is stored as a separate Run.

Action                                  Becomes a Run?
LLM call                                  Yes
Retriever / Vector search,                Yes
"Tool call (calculator, API)",            Yes
Chain / Node execution,                   Yes
Reranker call,                            Yes
Custom function (if traced),              Yes

It records unique ID, name, and type (for example, whether it is an LLM call, tool call, retriever, or chain). It also keeps track of the parent run if this step was called by another step, which helps build the hierarchy of execution.
Every Run also stores the input that was sent to that step and the output that it produced. For an LLM, this means the prompt or messages going in and the model’s response coming out. For a retriever, it stores the query and the documents that were returned.


Trace = Full story of one user request
When a user asks a question, many steps happen in the background.
LangSmith groups all those steps into one Trace.

User asks:
“How many casual leaves do I get?”
Behind the scenes:

Retriever searches documents
Reranker improves results
LLM generates answer

All of this together = 1 Trace
Inside that Trace you will see multiple Runs.

Projects in LangSmith

A Project is like a folder that stores related traces.
It helps you organize your work.

Why Projects Exist
Without projects, all traces from different apps would get mixed together.
Projects keep things clean and separated.

How to Set a Project
In your environment variables:
LANGCHAIN_PROJECT=rag-dev
All traces from your code will now go into the rag-dev project.

Why Projects Are Useful

Separate dev / staging / prod
Separate different applications
Easier filtering and debugging
Cleaner monitoring
Better team collaboration

Sessions in LangSmith

A Session groups multiple traces that belong to the same conversation.It Track full conversation and Analyze user behavior

Example
User conversation:

“How many casual leaves do I get?”
“Can I carry them forward?”
“What about sick leave?”

Each question creates a Trace.
All three traces together belong to one Session.

Concept         Meaning
Run            One single step
Trace          One full request/question
Session        One full conversation
Project        Folder for an app/environment

How tracing works

Tracing means automatically recording everything that happens when your LLM application runs.
When tracing is enabled, LangSmith captures each step and stores it as Runs and Traces.

1. You enable tracing (using env variables)
2. Your LangChain / LangGraph code runs
3. LangSmith automatically tracks every step
4. Data is sent to LangSmith cloud
5. You can see the full Trace in the dashboard

What Happens Behind the Scenes

Your code calls an LLM / Tool / Retriever,
LangSmith SDK wraps that call,
It records:
Input,
Output,
Time taken,
Tokens,
Errors,

This becomes a Run,
Multiple Runs are grouped into a Trace,
The Trace is stored under your Project

To enable tracing only set environment variables. No major code changes needed for basic tracing.



Enabling Tracing in LangChain / LangGraph

1. Environment-Based Configuration
Never hardcode values. Always use environment variables.


env# Production
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=your_prod_api_key
LANGCHAIN_PROJECT=rag-prod


env# Development
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=your_dev_api_key
LANGCHAIN_PROJECT=rag-dev
Use different keys and projects for each environment.


2. Centralized Config Setup

Create a config file (recommended):

import os
from dotenv import load_dotenv

load_dotenv()

LANGSMITH_CONFIG = {
    "tracing": os.getenv("LANGCHAIN_TRACING_V2", "false") == "true",
    "api_key": os.getenv("LANGCHAIN_API_KEY"),
    "project": os.getenv("LANGCHAIN_PROJECT", "default"),
    "environment": os.getenv("APP_ENV", "dev"),
    "app_version": os.getenv("APP_VERSION", "1.0.0"),
}


3. Always Attach Metadata (Industry Practice)

Pythondef get_run_config(user_id: str, session_id: str):
    return {
        "tags": [
            os.getenv("APP_ENV", "dev"),
            "rag",
            "production" if os.getenv("APP_ENV") == "prod" else "dev"
        ],
        "metadata": {
            "user_id": user_id,
            "session_id": session_id,
            "environment": os.getenv("APP_ENV", "dev"),
            "app_version": os.getenv("APP_VERSION", "1.0.0"),
            "service": "hr-knowledge-assistant"
        }
    }




  Practice                     Industry Standard?           Why

Separate projects for each env      Yes                  Clean separation
Different API keys per environment Yes                  Security
Metadata on every request       Yes                     Debugging
Tags for filtering              Yes                     Easy search
App version tracking            Yes                     Compare releases
Sampling in high traffic        Yes                     Cost control
Secrets in env / vault          Yes                     Security


1. Env variables for config
2. Separate LangSmith projects (dev/stage/prod)
3. Metadata: user_id, session_id, version, environment
4. Tags for filtering
5. Trace most requests (or sample if very high traffic)
6. Monitor errors + latency + cost


What is Trace View?

When you open a Trace in LangSmith, you see a detailed page that shows the complete execution of one request.
This is the main screen you use for debugging.

Shows high-level information:

Input (user question),
Output (final answer),
Total latency,
Total tokens,
Status (success / error),
Tags and metadata

This gives a quick overview.

When you open the Trace, it looks something like this:

Trace: rag_chain
├── Status: Success
├── Latency: 2.4 seconds
├── Total Tokens: 1,280
│
├── 1. Retriever
│     ├── Input: "How many casual leaves do employees get?"
│     ├── Output: 5 document chunks
│     └── Latency: 0.4s
│
├── 2. Reranker
│     ├── Input: 5 chunks
│     ├── Output: Top 3 ranked chunks
│     └── Latency: 0.6s
│
└── 3. ChatOpenAI (LLM)
      ├── Input: System prompt + Context + Question
      ├── Output: "Employees are entitled to 12 days of casual leave..."
      ├── Tokens: 820 input / 60 output
      └── Latency: 1.4s


Step           What You Find                      Conclusion
Retriever    Wrong documents retrieved            Retrieval problem
Reranker     Correct docs retrieved but ranked low    Reranking problem
LLM          Good context but poor answer,Prompt problem

What is Latency?

Latency = How much time a step (or full request) takes to complete.
In LangSmith, you can see latency for:

Full Trace
Each individual Run

Step-by-step approach:

Open the Trace
Check total latency
Expand all Runs
Compare time taken by each step
Identify the slowest Run
Decide how to optimize it

Common Bottlenecks:-

Slow Step             Possible Reason                                      Possible Fix

Retriever       Large vector index / no filter               "Add metadata filters, optimize index"
Reranker        Too many documents sent                      Reduce top_k before reranking
LLM             Large prompt / big context                  "Reduce context size, use smaller model"
Tool Call       Slow external API                            Optimize API / add caching
Multiple LLM calls  Agent doing too many steps               Improve planning / reduce loops

What is Token Usage?

Tokens are the pieces of text that LLMs read and generate.

Input tokens (Prompt tokens): Text you send to the model
Output tokens (Completion tokens): Text the model generates
Total tokens: Input + Output

More tokens = Higher cost + Sometimes higher latency

Reason                                Explanation

Cost control                    LLM cost depends on tokens
Performance optimization       Large prompts slow things down
Detect inefficient prompts     Find where tokens are wasted
Compare versions              See if new prompt uses fewer tokens

here You See Token Usage in LangSmith

Trace level → Total tokens for the full request
LLM Run level → Tokens used by that specific model call

Trace Total Tokens: 1,450

LLM Run:
- Prompt tokens: 1,200
- Completion tokens: 250
- Total: 1,450

What You Should Analyze

Metric                       What It Tells You

High prompt tokens        Context/prompt is too large
High completion tokens    Model is generating long answers
Sudden token increase     Something changed in prompt/retrieval
Tokens vs quality         Are extra tokens actually helping?

Case: Cost suddenly increased

You open LangSmith and find:

Old Trace:
Prompt tokens: 600
Completion tokens: 120

New Trace:
Prompt tokens: 1800
Completion tokens: 130

Conclusion:
Retrieval is sending too much context now.

Fix:
Reduce top_k
Improve context packing
Compress retrieved chunks

Common Reasons for High Token Usage:-

    Cause                          Solution
Too many retrieved chunks       Reduce top_k / better filtering
Large system prompt             Shorten system instructions
No context compression          Add compression step
Agent making many LLM calls     Reduce unnecessary steps
Sending full history always     Use summary memory instead

Best Practices

Track average tokens per request in production
Set internal limits (example: max context tokens)
Compare token usage when changing prompts
Optimize retrieval before increasing model size
Monitor cost along with token count

What is Error Debugging with Traces?

When something fails or gives a wrong answer, you use the Trace to find:

Where the error happened
Why it happened
What input caused it

Instead of guessing, you can see the exact failure point.

Types of Problems You Can Debug

Problem Type             Example

Hard failure        "Tool error, API error, exception"
Soft failure        "Wrong answer, hallucination"
Partial failure      Retrieval worked but generation failed
Agent failure        Wrong tool selected



Step-by-Step Debugging Process
1. Open the Failed Trace
Look for traces with:
Status = Error
Or wrong final output

2. Check the Overall Trace
Final input
Final output
Error message (if any)

3. Inspect Each Run
Go step by step:
textRetriever → Reranker → LLM → Tool
Find the first place where things went wrong.

4. Analyze Input and Output of That Run
Ask:
Was the input correct?
Was the output unexpected?
Did it throw an error?

5. Fix the Root Cause
Don’t just fix the final answer — fix the actual failing step.


Real Example 1:'
Hard Error
Problem: Application crashed
Tool Run: search_database
Status: Error
Error: Connection timeout
Conclusion: Database tool failed.
Fix: Handle timeout + retry logic.


Real Example 2: Wrong Answer
Problem: Bot gave incorrect leave policy
Trace inspection:
Step              Finding                           Result
Retriever     Retrieved wrong documents           Root cause
Reranker       Ranked wrong docs highly             Secondary
LLM           Generated answer from wrong context     Expected

Fix: Improve retrieval / filtering, not the LLM prompt.

Real Example 3: Agent Problem

Problem: Agent gave wrong final response
Trace shows:
 1. LLM decided to call calculator
2. Calculator got wrong input
3. LLM used wrong result

Root cause: Bad tool input formatting.
Fix: Improve tool argument generation.

Tool call inspection (important for agents)

What is Tool Call Inspection?

In agents, the LLM often decides to call external tools like:
Search,
Calculator,
Database,
API,
RAG retriever,

Tool Call Inspection means checking:

Which tool was selected,
What arguments were passed,
What the tool returned,
How the LLM used that result

This is critical because many agent failures happen at the tool level.

What You Can See in LangSmith for a Tool Call

When you open a Tool Run, you can inspect:

Tool name,
Input arguments,
Tool output,
Latency,
Error (if any),
Parent LLM decision

Why Filtering & Searching is Important

In production, you may have thousands of traces.
Without filters, finding the right one is very hard.
Filtering helps you quickly find:

Failed requests,
Specific user issues,
Slow requests,
Traces from a particular version,
A certain tool or chain


Main Ways to Filter Traces
1. Filter by Status

Success
Error

Useful for debugging failures quickly.

2. Filter by Time

Last 1 hour,
Last 24 hours,
Custom date range,

Useful for investigating recent issues.

3. Filter by Project
Example:

rag-prod,
rag-dev,

Keeps environments separate.

4. Filter by Tags
Examples:

prod,
experiment-v2,
hr-assistant,
error,

Tags are one of the most useful filters.

5. Filter by Metadata
Examples:

user_id = 123,
app_version = 1.4.0,
environment = prod,
session_id = abc,

Very powerful for production debugging.

6. Filter by Latency
Find traces that are:

Slower than 3 seconds,
Slower than 5 seconds,

Useful for performance analysis.

7. Full-Text Search
Search inside:

Inputs,
Outputs,
Error messages,

Example:
Search "casual leave" to find related traces.

What are Datasets in LangSmith?

A Dataset in LangSmith is a collection of test examples used to evaluate your LLM application.
Each example usually contains:

Input (question)
Expected output (reference answer)
Optional metadata

{
  "inputs": {
    "question": "How many casual leaves do employees get?"
  },
  "outputs": {
    "answer": "Employees get 12 days of casual leave per year."
  },
  "metadata": {
    "category": "leave_policy",
    "difficulty": "easy"
  }
}

Type                   Description                          Common Use
Manual Dataset       Created by humans                  High quality
Production Dataset  Built from real user traces      Realistic testing
Synthetic Dataset   Generated by LLM,Quick start
Hybrid Dataset      Mix of real + synthetic,Best approach

There are two main ways to create datasets:

Manual (from dashboard) and
Programmatic (using code)


1. Manual Dataset Creation (Dashboard)
Steps:

Open LangSmith → go to Datasets & Experiments,
Click New Dataset,
Enter details:-
Name: hr-policy-eval,
Description: optional,

Create the dataset
Click Add Example,
Fill:,
Input,

{
  "question": "How many casual leaves do employees get?"
}

Expected Output,

{
  "answer": "Employees are entitled to 12 days of casual leave every year."
}

Save

You can also upload CSV/JSON in many cases.

Programmatic Dataset Creation (Using Code)

This is the preferred way for serious projects.

Install (if needed)

pip install langsmith

Create Dataset + Add Examples

In [ ]:
from langsmith import Client

client = Client()

# 1. Create dataset
dataset = client.create_dataset(
    dataset_name="hr-policy-eval",
    description="Evaluation dataset for HR policy RAG"
)

# 2. Add examples
examples = [
    {
        "inputs": {"question": "How many casual leaves do employees get?"},
        "outputs": {"answer": "Employees are entitled to 12 days of casual leave every year."}
    },
    {
        "inputs": {"question": "Can casual leave be carried forward?"},
        "outputs": {"answer": "No, casual leave cannot be carried forward to the next year."}
    },
    {
        "inputs": {"question": "What is the sick leave policy?"},
        "outputs": {"answer": "Employees are entitled to 10 days of sick leave per year."}
    }
]

client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

print("Dataset created successfully")

3. Create Dataset from Production Traces
Very useful industry method:

Go to Traces

Select good/bad examples

Add them to a dataset

This helps create realistic evaluation data from actual user queries.

Summary

Manual: Create dataset in UI and add examples one by one

Programmatic: Use Client().create_dataset() and create_examples()

Best approach: Start manual for understanding, then move to programmatic

What is an Example?
An Example is one test case inside a Dataset.
Each example usually contains:

inputs → what you send to your app
outputs → expected / reference answer
metadata → optional extra info

{
  "inputs": {
    "question": "How many casual leaves do employees get?"
  },
  "outputs": {
    "answer": "Employees are entitled to 12 days of casual leave every year."
  },
  "metadata": {
    "category": "leave_policy",
    "difficulty": "easy"
  }
}

Good Practices for Examples

Keep expected answers short and factual
Use consistent field names (question, answer)
Add metadata for category/difficulty
Include edge cases:
Questions with no answer in docs
Ambiguous questions
Multi-part questions

Avoid very long reference answers


What is a Schema?

A Schema defines the structure of inputs and outputs for a dataset.
It makes sure all examples follow the same format.
Schema = Structure/rules for those examples

inputs:
  - question: string

outputs:
  - answer: string

  This ensures consistency across all examples.

What Does Running Evaluation Mean?

It means:

Take examples from your Dataset
Run your RAG / Agent on each example
Score the results using evaluators
Store and analyze the scores

This helps you measure how good your system is.

Dataset (Questions + Expected Answers)
        ↓
Your App (RAG / Agent)
        ↓
Actual Answers
        ↓
Evaluators (Scoring)
        ↓
Results in LangSmith

Evaluation function:-

Parameter     Meaning

run_ran      Your application function
data        Dataset name or ID
evaluators   Scoring functions
experiment_prefix  Name for this evaluation run

In [ ]:

#Basic Code to Run Evaluation

from langsmith import evaluate

def run_rag(inputs: dict) -> dict:
    question = inputs["question"]
    
    # Your RAG logic here
    answer = your_rag_pipeline(question)
    
    return {"answer": answer}

results = evaluate(
    run_rag,
    data="hr-policy-eval",          # dataset name
    evaluators=[...],               # list of evaluators
    experiment_prefix="rag-v1"
)


What Happens When You Run Evaluation?
For each example in the dataset:

LangSmith sends the input to your function
Your app generates an answer
Evaluators compare:
Predicted answer
Expected answer
(Sometimes) context

Scores are saved under an Experiment

Where to See Results
In LangSmith dashboard:

Go to Datasets & Experiments
Open your dataset
Open the experiment (rag-v1, rag-v2, etc.)
See:
Overall scores
Per-example results
Failures
Latency / tokens

ractical Example
Dataset has 3 questions:

Casual leave days
Carry forward policy
Sick leave days

You run evaluation with rag-v1.
Results might look like:

Example       Score   Status
Casual leave  1.0     Pass
Carry forward  0.0    Fail
Sick leave    1.0     Pass

Overall score: 66% Then you improve retrieval and run rag-v2.


Running evaluation means:

Execute your app on a dataset
Score outputs using evaluators
Analyze results in LangSmith
Improve and repeat

This is the core feedback loop for improving RAG and Agents.

What are Built-in Evaluators?

Built-in evaluators are ready-made scoring functions provided by LangSmith / LangChain.
You can use them directly without writing your own evaluation logic.

Common Built-in Evaluators:-

(1)Correctness:- Is the answer factually correct? It used for General Q&A
(2)Contextual Q&A / Faithfulness:- It checked Is answer correct based on retrieved context?  and used for RAG systems.

Start with built-in evaluators first:
Then add custom evaluators only when needed.

What is a Custom Evaluator?

A Custom Evaluator is a scoring function you write yourself.
Use it when built-in evaluators are not enough.
Example cases:

Check if answer contains a citation
Check if answer length is within limit
Check domain-specific rules
Custom business logic

Basic Structure of a Custom Evaluator
A custom evaluator is usually a function that:

Receives the run / example data
Applies your logic
Returns a score (and optional comment)

In [ ]:
#SAnother Example: Answer Length Check


def answer_length_check(run: Run, example: Example):
    answer = run.outputs.get("answer", "")
    word_count = len(answer.split())
    
    # Prefer answers between 10 and 80 words
    score = 1.0 if 10 <= word_count <= 80 else 0.0
    
    return {
        "key": "answer_length",
        "score": score,
        "comment": f"Word count: {word_count}"
    }



When Should You Create Custom Evaluators?

Situation                            Use Custom Evaluator?
Need business-specific rules            Yes
"Need format checks (JSON, citations)"   Yes
Built-in evaluators already enough          No
Complex quality judgment            Prefer LLM-as-judge

Best Practices

Keep evaluators simple and focused

Return clear score (0 to 1)
Add useful comments
Name the key clearly (has_citation, correct_policy)
Don’t put too much logic in one evaluator

What is LLM-as-Judge?

LLM-as-Judge means using one LLM to evaluate the output of another LLM (or your RAG/Agent).
Instead of only using exact match or rules, you ask a strong model to judge quality.
Example:

Your RAG system generates an answer
Another LLM checks if the answer is correct, helpful, or faithful to context

What It Can Evaluate

Correctness,
Faithfulness (grounding),
Relevance,
Helpfulness,
Conciseness,
Toxicity / safety,
Instruction following

1. Dataset example has question + expected answer
2. Your RAG generates actual answer
3. Judge LLM receives:
   - Question
   - Expected answer (optional)
   - Actual answer
   - Context (optional)
4. Judge returns score + reasoning

Best Practices

Use a strong model as judge (example: GPT-4o)
Keep judge prompts clear and strict
Ask for both score and reason
Use multiple examples before trusting scores
Combine with rule-based evaluators when possible

LLM-as-Judge = Use an LLM to score another model’s output.
It is one of the most powerful evaluation methods in LangSmith, especially for:

RAG quality
Agent responses
Open-ended generation

Most Used Evaluation Methods in Production

In real production systems, teams usually don’t rely on only one method.
They combine methods based on the stage.

What is Most Used?

Rank        Method                    Usage in Production      Best For
1        Hybrid (Code + LLM-as-Judge)    Most common        Overall quality
2        LLM-as-Judge                    Very common,     "Correctness, faithfulness, helpfulness"
3        Code / Rule-based evaluators     Very common,        "Format, citations, schema, keywords"
4        User feedback,                       Common            Real-world signal
5        Human annotation               Used selectively       High-risk cases

Why Version Comparison is Important

When you change something in your system:

Prompt,
Model,
Retrieval strategy,
Reranker,
Agent logic,

You need to know:
Did the new version become better or worse?
LangSmith helps you compare versions systematically.

Same Dataset
   ↓
Run Version A (example: rag-v1)
   ↓
Run Version B (example: rag-v2)
   ↓
Compare scores side by side

Keep the dataset fixed. Change only the system version.

Regression Testing in LangSmith

What is Regression Testing?
Regression testing means checking that a new change did not break things that were already working.
In simple words:
“After my update, is the system still at least as good as before?”

Why Regression Testing is Important
When you improve one thing, you might accidentally make something else worse.
Examples:

New prompt improves style but reduces factual accuracy
Adding reranking improves some answers but breaks others
Agent update causes more tool failures

Regression testing protects you from these silent quality drops.

1. Keep a fixed evaluation dataset
2. Record baseline scores (old version)
3. Make your change
4. Run evaluation again (new version)
5. Compare against baseline
6. Pass only if quality does not drop

Practical Example
Baseline (rag-v1):

Correctness: 0.80
Faithfulness: 0.85

After change (rag-v2):

Correctness: 0.74
Faithfulness: 0.82

Result:

This is a regression
Even if some answers look better, overall quality droppedś

What is Prompt Hub?

Prompt Hub is a central place in LangSmith to:

Store prompts
Version prompts
Share prompts
Reuse prompts across applications

Think of it as GitHub for prompts.

Why Prompt Hub Exists
Without Prompt Hub:

Prompts are hardcoded in code
Hard to track changes
Hard to share with team
Hard to test different prompt versions

With Prompt Hub:

Prompts are managed centrally
Every change is versioned
Easy to pull into code
Easy to compare prompt versions

1. Create prompt in LangSmith UI / code
2. Save it to Prompt Hub
3. Pull it into your application
4. Update prompt → new version created
5. Compare versions using evaluation

Why It’s Useful in Production

No more hardcoding prompts
Safe updates (version control)
Easy rollback if new prompt performs worse
Team collaboration
Better experiment tracking



What is Prompt Versioning?

Prompt versioning means saving every change to a prompt as a new version, so you can:

Track what changed
Compare performance of different versions
Roll back if a new prompt performs worse
Use a stable prompt version in production


Why Prompt Versioning is Important
Without versioning:

You overwrite prompts
You don’t know which prompt produced a result
Hard to reproduce old behavior
Risky to update production prompts

With versioning:

Every change is recorded
You can test safely
You can roll back quickly
Production stays stable


Prompt: hr-rag-system-prompt

v1 → Basic prompt
v2 → Added grounding rules
v3 → Improved citation format
v4 → Reduced verbosity

Each save creates a new version/commit.

Practical Workflow

Create prompt in Prompt Hub
Save as v1
Evaluate system quality
Improve prompt
Save as v2
Run evaluation again
Compare v1 vs v2
Keep the better one in production

Recommended Process (Industry Style)
1. Develop prompt in playground
2. Save version
3. Evaluate on dataset
4. If better → promote to staging
5. Re-test
6. Pin version in production
7. Monitor
8. Repeat for next improvement

What is a Prompt Template?

A Prompt Template is a reusable prompt with variables.

System: You are an HR assistant. Answer only from context.
User: Question: {question}
Context: {context}

Here, {question} and {context} are variables.

Why Manage Prompt Templates?
If prompts are scattered in code:

Hard to update
Hard to share
Hard to version
Hard to test

Managing them in LangSmith helps you keep prompts organized, versioned, and reusable.

In real projects you usually have multiple environments:

Dev → experimentation
Staging → pre-production testing
Production → live users

You should not use the same prompt handling strategy in all three.

Recomended stratgy:-

Environment,              Prompt Usage Strategy,                Risk Level
Dev,                     Use latest / experimental prompts,   High experimentation
Staging,                Use candidate stable versions,          Medium
Production,              Use pinned tested versions only,       Low risk


1. Build/improve prompt in Dev
2. Evaluate on dataset
3. Promote good version to Staging
4. Re-test in Staging
5. Pin that version in Production
6. Monitor
7. Repeat

Example Promotion Path

Dev:
hr-rag-system-prompt:v5 (testing new grounding rules)

Staging:
hr-rag-system-prompt:v5 (candidate)

Production:
hr-rag-system-prompt:v4 (current stable)

After staging success:
Production switches to v5

User feedback collection

When a user gives feedback (👍 / 👎 / rating / comment), that feedback is attached to the related Trace in LangSmith.
So the Trace stores:

Input
Output
Intermediate steps
Feedback score
Feedback comment (optional)

This makes it easy to analyze quality directly from traces.Now in LangSmith, that Trace shows feedback clearly.

What are Annotation Queues?

Annotation Queues are a structured way to send selected traces/runs to human reviewers for quality checking.
Instead of randomly opening traces, reviewers get a clean queue of items to score.

Why Annotation Queues Exist
Automated evaluators are useful, but not perfect.
Human review is needed for:

Edge cases
Ambiguous answers
High-risk domains (HR, legal, medical, finance)
Improving evaluation datasets

Annotation queues make this human review organized.

1. Select important traces (errors, low feedback, random sample)
2. Add them to an Annotation Queue
3. Reviewers score each item using a rubric
4. Feedback is stored on the run/trace
5. Use results to improve system + dataset

What is Online Evaluation?

Online evaluation means evaluating your system on real production traffic, not just on a fixed offline

Offline evaluation  →  Test on prepared dataset
Online evaluation   →  Score live user requests

You should evaluate at least:

Correctness
Faithfulness / Grounding
Relevance
Citation present or not
Latency
User feedback

User request comes in
   ↓
Trace is created
   ↓
Evaluator runs on that trace (or a sample)
   ↓
Score is attached to the trace
   ↓
You monitor scores over time

You usually should not evaluate 100% of production traffic with expensive LLM judges.
Common approach:

100% with cheap code evaluators
5–20% with LLM-as-Judge
100% of negative feedback / errors

Why Cost Tracking Matters
LLM systems can become expensive quickly because of:

Large prompts
Too much retrieved context
Multiple agent steps
High traffic
Expensive models

If you don’t track cost, it grows silently.
What to Track

Cost per request
Token usage per request
Cost by model
Cost by feature / endpoint
Daily and monthly total cost
Cost by app version

In LangSmith, token usage from traces helps estimate this.

Problem                                   Why Cost Increases

Sending too many chunks to LLM,           Large prompt tokens
No context packing,                         Wasted tokens
Agent making unnecessary tool/LLM calls,    Extra generations
Using strong model for all tasks,           Overkill for simple questions
Evaluating 100% traffic with LLM judge,      Expensive online eval


1. Track average cost/request
2. Break down by model and feature
3. Find top expensive traces
4. Identify why they are expensive
5. Optimize those patterns
6. Re-measure

Why Sampling is Needed

If your system has high traffic, tracing and evaluating every request can become:

Expensive
Noisy
Hard to manage

Sampling means you capture only a portion of traffic intelligently.

Common Sampling Strategies
1. Random Sampling
Trace/evaluate a fixed percentage of requests.
Example:

10% of all production requests

Pros: Simple
Cons: May miss important edge cases

2. Error-Priority Sampling (Recommended)

100% of errors
100% of low feedback
5–20% of successful requests

This gives best visibility with lower cost.



CI/CD Integration with LangSmith

CI/CD integration means automatically testing your LLM app during the development pipeline, using LangSmith evaluations, before code goes to production.

Code change
   ↓
CI pipeline runs
   ↓
LangSmith evaluation on dataset
   ↓
Pass/Fail gate
   ↓
Deploy (only if pass)

Typical CI/CD Flow with LangSmith

Developer changes prompt / retrieval / agent logic
Pull Request is created
CI job runs evaluation on a fixed dataset
LangSmith stores experiment results
Pipeline checks scores against thresholds
If scores are good → merge/deploy
If scores drop → block release

Deploy only if:
- Correctness >= baseline - 2%
- Faithfulness >= baseline
- Error rate == 0 on smoke tests
- p95 latency within accepted limit

Practical CI Pipeline Stages

1. Unit tests (normal code tests)
2. Build app
3. Run LangSmith evaluation (dataset)
4. Compare with baseline scores
5. Pass/Fail decision
6. Deploy to staging/production

Summary
CI/CD + LangSmith means:

Automatically evaluate every important change
Compare against baseline
Block bad releases
Keep quality stable as system evolves

This is a key architect-level practice for serious LLM systems.

LangSmith vs other observability tools (Phoenix, TruLens, etc.)

LangSmith → Best all-round choice for LangChain/LangGraph production systems

Langfuse → Best open-source alternative

Braintrust → Best evaluation-first platform

Phoenix → Best OTel/self-host observability toolkit

For your path, mastering LangSmith is the right focus.